# Extract places from biographies

Claude Code agents read the biography of each benchmark individual (`01_benchmark_creation.ipynb`), blind to Cultura, and extract the place and the years of each life stage: pre-active, active and post-active life. Agent outputs are cached in `benchmark300/extraction_life_stages/`: an individual already extracted is not sent again. The places are associated with Cliopatria polities in `03_cliopatria_association.ipynb`.

In [ ]:
import json
import subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import polars as pl
from IPython.display import Markdown
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent
FOLDER = ROOT / "scripts" / "5-annotations" / "benchmark" / "benchmark300"
PROMPTS = ROOT / "notebooks" / "notebooks_validation" / "prompts"
EXTRACTION = FOLDER / "extraction_life_stages"
LOGS = FOLDER / "logs"
MODEL = "claude-opus-5-5"
STAGES = ["pre_active", "active", "post_active"]
EXTRACTION_GROUP = 10
WORKERS = 10
LIMIT = None
for folder in (EXTRACTION, LOGS):
    folder.mkdir(exist_ok=True)

## Benchmark individuals

The individuals whose biography could be downloaded (`building_scripts/benchmark300_texts.py`), one text file each in `benchmark300/texts/`.

In [ ]:
people = pl.DataFrame([{key: person[key] for key in ("qid", "label", "dictionary_name", "biography_url")}
                       for person in json.loads((FOLDER / "benchmark300_texts.json").read_text(encoding="utf-8"))])
if LIMIT:
    people = people.head(LIMIT)
assert people["qid"].is_unique().all()
assert all((FOLDER / "texts" / f"{qid}.txt").exists() for qid in people["qid"])
print(f"{len(people)} individuals")
people.head()

### Claude Code runner

Each agent is a headless Claude Code session (`claude -p`) started from the repository root, with only the tools it needs; its full output is kept in `benchmark300/logs/`.

In [ ]:
def claude(prompt, tools, allowed, log):
    """Run one headless Claude Code session and return its final reply."""
    command = ["claude", "-p", "--model", MODEL, "--output-format", "json", *(["--allowedTools", *allowed] if allowed else []), "--tools", *tools]
    result = subprocess.run(command, input=prompt, cwd=ROOT, capture_output=True, text=True, timeout=3600)
    log.write_text(result.stdout + result.stderr, encoding="utf-8")
    return json.loads(result.stdout)["result"]


def groups(qids, size):
    return [qids[start:start + size] for start in range(0, len(qids), size)]


def done(path, qids):
    return path.exists() and set(qids) <= set(json.loads(path.read_text(encoding="utf-8")))


def run_all(task, jobs, label):
    """Run task(number, qids) for every job in parallel; a failed group is reported and left for the next run."""
    def safe(job):
        try:
            task(*job)
        except Exception as error:
            print(f"{label} group {job[0]:02d} failed: {error}")
    with ThreadPoolExecutor(WORKERS) as pool:
        list(tqdm(pool.map(safe, jobs), total=len(jobs), desc=label))

## Extraction prompt

In [ ]:
EXTRACTION_PROMPT = (PROMPTS / "biography_life_stage_extraction.md").read_text(encoding="utf-8").split("---", 1)[1]
Markdown(EXTRACTION_PROMPT)

### Run the extraction agents

Groups of `EXTRACTION_GROUP` individuals, `WORKERS` agents at a time; individuals already extracted by an earlier run are not sent again. Each agent reads, greps and writes files, and may run `.venv/bin/python` to check its output.

In [ ]:
def extract(number, qids):
    path = EXTRACTION / f"agent_{number:03d}.json"
    if not (done(path, qids) or set(qids) <= already_extracted):
        prompt = EXTRACTION_PROMPT.replace("{QIDS}", " ".join(qids)).replace("{N}", f"{number:03d}")
        claude(prompt, ["Read", "Grep", "Glob", "Write", "Bash"], ["Read", "Grep", "Glob", "Write", "Bash(.venv/bin/python:*)"], LOGS / f"extraction_{number:03d}.log")


def extracted():
    return {qid for path in EXTRACTION.glob("agent_*.json") for qid in json.loads(path.read_text(encoding="utf-8"))}


already_extracted = extracted()
extraction_jobs = list(enumerate(groups(people["qid"].to_list(), EXTRACTION_GROUP)))
run_all(extract, extraction_jobs, "extraction")
already_extracted = extracted()
missing = [number for number, qids in extraction_jobs if not set(qids) <= already_extracted]
print(f"{len(extraction_jobs) - len(missing)} / {len(extraction_jobs)} groups extracted; missing: {missing}")

### Extracted places

Individuals extracted, and stages with a place and with years.

In [ ]:
qids = set(people["qid"])
extraction = {qid: person for path in sorted(EXTRACTION.glob("agent_*.json"))
              for qid, person in json.loads(path.read_text(encoding="utf-8")).items() if qid in qids}
print(f"{len(extraction)} / {len(people)} individuals extracted")
filled = pl.DataFrame([{"stage": stage, "place": bool(person.get(stage)),
                        "years": bool(person.get(stage) and any(year is not None for year in person[stage].get("years") or []))}
                       for person in extraction.values() for stage in STAGES])
filled.group_by("stage").agg(individuals=pl.len(), with_place=pl.col("place").sum(), with_years=pl.col("years").sum()).sort(pl.col("stage").cast(pl.Enum(STAGES)))

In [ ]:
pl.DataFrame([{"qid": qid, "stage": stage, "place": person[stage]["place"], "years": person[stage].get("years"), "confidence": person[stage]["confidence"]}
              for qid, person in extraction.items() for stage in STAGES if person.get(stage)]).sample(8, seed=42)